<a href="https://colab.research.google.com/github/guilhermemoraes-lasalle/ColecoesEassociacoes/blob/main/Exercicio_Cruzamento_Limpeza_Vendas_Clientes_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercício Prático — Limpeza, Cruzamento e Análise de Vendas

Neste notebook serão trabalhadas duas bases de um comércio eletrônico:

- **Vendas:** histórico de transações, valores, categorias, datas, status e e-mails.
- **Clientes:** cadastro dos clientes com nome e cidade.

O objetivo é realizar diagnóstico, tratamento de valores ausentes, filtros, cruzamento de bases, transformações e análise temporal utilizando **Pandas** e **NumPy**.


## Configuração do Ambiente e Criação dos DataFrames


In [1]:
import numpy as np
import pandas as pd

# Base de vendas
dados_vendas = {
    'cliente_id': [101, 102, 103, 101, 104, 102, 105, 103],
    'valor': [3500.75, 189.50, np.nan, 1200.00, 450.00, np.nan, 89.90, 780.50],
    'categoria': [
        'Eletronicos',
        'Livros',
        'Roupas',
        'Eletronicos',
        'Automotivo',
        'Livros',
        'Roupas',
        'Roupas',
    ],
    'data_hora': [
        '2024-01-15 10:23:00',
        '2024-01-18 14:05:00',
        '2024-02-05 09:12:00',
        '2024-02-20 16:40:00',
        '2024-03-02 11:00:00',
        '2024-03-15 18:30:00',
        '2024-04-10 08:20:00',
        '2024-04-22 13:45:00',
    ],
    'status': [
        'Concluído',
        'Concluído',
        'Pendente',
        'Concluído',
        'Cancelado',
        'Concluído',
        'Concluído',
        'Concluído',
    ],
    'email': [
        'maria@gmail.com',
        'joao@outlook.com',
        'ana@yahoo.com',
        'maria@gmail.com',
        'carlos@gmail.com',
        'joao@outlook.com',
        'lucas@empresa.com.br',
        'ana@yahoo.com',
    ],
}

# Base de clientes
dados_clientes = {
    'cliente_id': [101, 102, 103, 104, 105],
    'nome': [
        'Maria Silva',
        'Joao Souza',
        'Ana Oliveira',
        'Carlos Lima',
        'Lucas Mendes',
    ],
    'cidade': [
        'Sao Paulo',
        'Rio de Janeiro',
        'Belo Horizonte',
        'Curitiba',
        'Salvador',
    ],
}

df_vendas = pd.DataFrame(dados_vendas)
df_clientes = pd.DataFrame(dados_clientes)

print("DataFrame de vendas:")
display(df_vendas)

print("\nDataFrame de clientes:")
display(df_clientes)


DataFrame de vendas:


,cliente_id,valor,categoria,data_hora,status,email
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com
1,102,189.50,Livros,2024-01-18 14:05:00,Concluído,joao@outlook.com
2,103,NaN,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com
4,104,450.00,Automotivo,2024-03-02 11:00:00,Cancelado,carlos@gmail.com
5,102,NaN,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com
6,105,89.90,Roupas,2024-04-10 08:20:00,Concluído,lucas@empresa.com.br
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com



DataFrame de clientes:


,cliente_id,nome,cidade
0,101,Maria Silva,Sao Paulo
1,102,Joao Souza,Rio de Janeiro
2,103,Ana Oliveira,Belo Horizonte
3,104,Carlos Lima,Curitiba
4,105,Lucas Mendes,Salvador


## Parte 1 — Diagnóstico e Limpeza


### 1. Dimensões e tipos de dados


In [2]:
print("Dimensões de df_vendas:", df_vendas.shape)

print("\nInformações do DataFrame:")
df_vendas.info()


Dimensões de df_vendas: (8, 6)

Informações do DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   cliente_id  8 non-null      int64  
 1   valor       6 non-null      float64
 2   categoria   8 non-null      object 
 3   data_hora   8 non-null      object 
 4   status      8 non-null      object 
 5   email       8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


### 2. Identificação e tratamento dos valores nulos


In [3]:
print("Valores nulos por coluna:")
print(df_vendas.isnull().sum())

print("\nRegistros com valor ausente:")
display(df_vendas[df_vendas['valor'].isna()])


Valores nulos por coluna:
cliente_id    0
valor         2
categoria     0
data_hora     0
status        0
email         0
dtype: int64

Registros com valor ausente:


,cliente_id,valor,categoria,data_hora,status,email
2,103,NaN,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com
5,102,NaN,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com


Neste exercício, os valores ausentes serão preenchidos preferencialmente pela **mediana da respectiva categoria**. Caso alguma categoria não possua uma mediana válida, será utilizada a mediana global como alternativa.


In [4]:
# Mediana de valor por categoria
mediana_categoria = (
    df_vendas.groupby('categoria')['valor']
    .transform('median')
)

# Mediana global
mediana_global = df_vendas['valor'].median()

# Primeiro tenta preencher pela categoria e depois pela mediana global
df_vendas['valor'] = (
    df_vendas['valor']
    .fillna(mediana_categoria)
    .fillna(mediana_global)
)

print("Mediana global:", mediana_global)

print("\nValores após o tratamento:")
display(df_vendas)

print("\nQuantidade de valores nulos restante:")
print(df_vendas['valor'].isnull().sum())


Mediana global: 615.25

Valores após o tratamento:


,cliente_id,valor,categoria,data_hora,status,email
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com
1,102,189.50,Livros,2024-01-18 14:05:00,Concluído,joao@outlook.com
2,103,435.20,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com
4,104,450.00,Automotivo,2024-03-02 11:00:00,Cancelado,carlos@gmail.com
5,102,189.50,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com
6,105,89.90,Roupas,2024-04-10 08:20:00,Concluído,lucas@empresa.com.br
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com



Quantidade de valores nulos restante:
0


### 3. Transações concluídas com valor superior a R$ 500


In [5]:
transacoes_filtradas = df_vendas.loc[
    (df_vendas['status'] == 'Concluído') &
    (df_vendas['valor'] > 500)
]

display(transacoes_filtradas)


,cliente_id,valor,categoria,data_hora,status,email
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com


## Parte 2 — Cruzamento e Transformação


### 4. Left merge entre vendas e clientes


In [6]:
df_completo = pd.merge(
    df_vendas,
    df_clientes,
    on='cliente_id',
    how='left'
)

display(df_completo)


,cliente_id,valor,categoria,data_hora,status,email,nome,cidade
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo
1,102,189.50,Livros,2024-01-18 14:05:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro
2,103,435.20,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com,Ana Oliveira,Belo Horizonte
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo
4,104,450.00,Automotivo,2024-03-02 11:00:00,Cancelado,carlos@gmail.com,Carlos Lima,Curitiba
5,102,189.50,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro
6,105,89.90,Roupas,2024-04-10 08:20:00,Concluído,lucas@empresa.com.br,Lucas Mendes,Salvador
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com,Ana Oliveira,Belo Horizonte


### 5. Média de vendas por categoria com `transform()`


In [7]:
df_completo['media_categoria'] = (
    df_completo.groupby('categoria')['valor']
    .transform('mean')
)

display(
    df_completo[
        ['cliente_id', 'categoria', 'valor', 'media_categoria']
    ]
)


,cliente_id,categoria,valor,media_categoria
0,101,Eletronicos,3500.75,2350.375
1,102,Livros,189.50,189.500
2,103,Roupas,435.20,435.200
3,101,Eletronicos,1200.00,2350.375
4,104,Automotivo,450.00,450.000
5,102,Livros,189.50,189.500
6,105,Roupas,89.90,435.200
7,103,Roupas,780.50,435.200


### 6. Clientes que utilizam Gmail


In [8]:
usuarios_gmail = df_completo[
    df_completo['email'].str.contains(
        '@gmail.com',
        case=False,
        na=False,
        regex=False
    )
]

# Como um cliente pode aparecer em várias vendas, contamos IDs únicos
quantidade_clientes_gmail = usuarios_gmail['cliente_id'].nunique()

print("Quantidade de clientes únicos que utilizam Gmail:",
      quantidade_clientes_gmail)

display(
    usuarios_gmail[
        ['cliente_id', 'nome', 'email']
    ].drop_duplicates()
)


Quantidade de clientes únicos que utilizam Gmail: 2


,cliente_id,nome,email
0,101,Maria Silva,maria@gmail.com
4,104,Carlos Lima,carlos@gmail.com


## Parte 3 — Análise Temporal e Agregação


### 7. Conversão de data e extração do mês e dia da semana


In [9]:
df_completo['data_hora'] = pd.to_datetime(
    df_completo['data_hora']
)

# Extraindo mês e dia da semana
df_completo['mes'] = df_completo['data_hora'].dt.month_name()
df_completo['dia_semana'] = df_completo['data_hora'].dt.day_name()

display(
    df_completo[
        ['data_hora', 'mes', 'dia_semana']
    ]
)


,data_hora,mes,dia_semana
0,2024-01-15 10:23:00,January,Monday
1,2024-01-18 14:05:00,January,Thursday
2,2024-02-05 09:12:00,February,Monday
3,2024-02-20 16:40:00,February,Tuesday
4,2024-03-02 11:00:00,March,Saturday
5,2024-03-15 18:30:00,March,Friday
6,2024-04-10 08:20:00,April,Wednesday
7,2024-04-22 13:45:00,April,Monday


### 8. Tabela dinâmica: categoria × cidade


In [10]:
tabela_dinamica = pd.pivot_table(
    df_completo,
    values='valor',
    index='categoria',
    columns='cidade',
    aggfunc='sum',
    fill_value=0
)

print("Total do valor de vendas por categoria e cidade:")
display(tabela_dinamica)


Total do valor de vendas por categoria e cidade:


cidade,Belo Horizonte,Curitiba,Rio de Janeiro,Salvador,Sao Paulo
categoria,,,,,
Automotivo,0.0,450.0,0.0,0.0,0.00
Eletronicos,0.0,0.0,0.0,0.0,4700.75
Livros,0.0,0.0,379.0,0.0,0.00
Roupas,1215.7,0.0,0.0,89.9,0.00


## Base Final Preparada


In [11]:
display(df_completo)


,cliente_id,valor,categoria,data_hora,status,email,nome,cidade,media_categoria,mes,dia_semana
0,101,3500.75,Eletronicos,2024-01-15 10:23:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo,2350.375,January,Monday
1,102,189.50,Livros,2024-01-18 14:05:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro,189.500,January,Thursday
2,103,435.20,Roupas,2024-02-05 09:12:00,Pendente,ana@yahoo.com,Ana Oliveira,Belo Horizonte,435.200,February,Monday
3,101,1200.00,Eletronicos,2024-02-20 16:40:00,Concluído,maria@gmail.com,Maria Silva,Sao Paulo,2350.375,February,Tuesday
4,104,450.00,Automotivo,2024-03-02 11:00:00,Cancelado,carlos@gmail.com,Carlos Lima,Curitiba,450.000,March,Saturday
5,102,189.50,Livros,2024-03-15 18:30:00,Concluído,joao@outlook.com,Joao Souza,Rio de Janeiro,189.500,March,Friday
6,105,89.90,Roupas,2024-04-10 08:20:00,Concluído,lucas@empresa.com.br,Lucas Mendes,Salvador,435.200,April,Wednesday
7,103,780.50,Roupas,2024-04-22 13:45:00,Concluído,ana@yahoo.com,Ana Oliveira,Belo Horizonte,435.200,April,Monday


## Conclusão

Foram realizadas as principais operações solicitadas no exercício:

- inspeção das dimensões e tipos de dados;
- identificação e tratamento de valores ausentes;
- filtragem de transações;
- junção entre as bases de vendas e clientes;
- cálculo da média por categoria com `groupby()` e `transform()`;
- identificação de clientes com Gmail usando métodos de texto;
- conversão e extração de informações temporais;
- construção de tabela dinâmica com `pivot_table()`.

A base final está pronta para ser utilizada em relatórios e análises gerenciais.
